# 01 · Clean the unified dataset

**Shared pipeline notebook (team-owned).** Run it after `00_build_dataset.ipynb`. Changes go through a PR.

**Input:** `data/clean/films_unified.csv` (one row per film, every source's columns side by side).
**Outputs** (gitignored; rebuilt on every run):
- `data/clean/films_clean.csv`: same films and rows, plus one **combined column per fact** (runtime, US release date, rating, genre…), with a `_src` column showing which source supplied the value and a `_missing` flag for inputs that aren't always filled.
- `data/clean/films_clean_columns.csv`: a dictionary giving every column's group and role.

**Why:** each fact (runtime, genre, rating…) comes from several sources, each in its own column. The rules, agreed in `Edwards-A10.ipynb` §4b (Tanner, 2026-11-24), are:
1. Each fact gets **one combined column**, filled from the fullest source first, then from the others in a fixed order.
2. Only real values count. TMDB's `0` for budget, revenue or runtime means unknown, and so does Box Office Mojo's `-` distributor.
3. Values are put into one format before combining (rating codes, genre names, language codes, country names).
4. The original source columns are **kept but grouped as `source detail`**: there for audits, never model inputs.
5. Inputs that aren't always filled get a `_missing` flag. Blanks aren't guessed or dropped, because coverage depends on budget (Data Understanding §3h).

**This notebook never removes films.** Films that fail a cleaning rule get an `exclude_reason`; `in_population` marks the films the model uses (contract amendment A7: released in the US, after the rules). §10b–12 add the rules Tanner decided in `Edwards-A10.ipynb` §8–9 (2026-11-24, agreed with Matt):
- §10b budget repair, §10c Wikidata fills, §10d `source_group`
- §11b rule-based fills (MPAA, country, franchise)
- §12 exclusion rules, population and the label (US box office < 1.25× budget)

Still to do: standardizing company and distributor names.

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

CLEAN = Path("../data/clean")
films = pd.read_csv(CLEAN / "films_unified.csv", low_memory=False)
assert films["film_uid"].is_unique
n_rows = len(films)
usable = films["usable"]
print(f"films_unified.csv: {n_rows:,} films x {films.shape[1]} columns ({usable.sum():,} usable)")

## 2. Helpers

- `real(col)`: the column with placeholder values turned into blanks.
- `combine(name, [cols])`: the first real value across the columns, in order, plus `<name>_src` naming the source column it came from.

In [ ]:
ZERO_MISSING = {"budget", "revenue", "runtime", "tn_Domestic Gross (USD)"}   # 0 = unknown in these sources
DASH_MISSING = {"bom_distributor"}


def real(c):
    s = films[c]
    if c in ZERO_MISSING:
        s = s.where(s.ne(0))
    if s.dtype == object or str(s.dtype) == "str":
        s = s.where(s.astype(str).str.strip().ne(""))
    if c in DASH_MISSING:
        s = s.where(s.astype(str).str.strip().ne("-"))
    return s


def combine(name, cols, values=None):
    """First real value across `cols` (in order). `values` optionally supplies already-formatted Series per col."""
    vals = pd.concat([(values or {}).get(c, real(c)).rename(c) for c in cols], axis=1)
    src = pd.Series(pd.NA, index=films.index, dtype="string")
    for c in reversed(cols):
        src = src.mask(vals[c].notna(), c)
    films[name] = vals.bfill(axis=1).iloc[:, 0].infer_objects() if len(cols) > 1 else vals.iloc[:, 0]
    films[name + "_src"] = src
    u = films.loc[usable, name]
    print(f"{name:<22} usable films filled {u.notna().mean():6.1%}   by source: "
          + ", ".join(f"{k} {v:.0%}" for k, v in films.loc[usable, name + '_src'].value_counts(normalize=True).items()))


def split_list(s, sep=","):
    return s.fillna("").astype(str).map(lambda x: [t.strip() for t in x.split(sep) if t.strip()])


def join_list(lists):
    return lists.map(lambda l: ", ".join(dict.fromkeys(l)) if l else np.nan)

## 3. Money (C1–C3)

`fin_budget` and `fin_revenue` are already combined by the build (TMDB → The Numbers → Wikidata) and are kept unchanged. `domestic_gross` is new: The Numbers → Box Office Mojo → RT. It's post-release, so it's **for checks only** (S4, U3), never an input. A The Numbers domestic gross of 0 counts as unknown.

In [ ]:
combine("domestic_gross", ["tn_Domestic Gross (USD)", "bom_domestic_gross", "rt_boxOffice_usd"])

## 4. Runtime and release date (C4–C6)

- `runtime_min`: TMDB → RT → The Numbers → RT sl (sources agree on 92–93% of films within 5 minutes).
- `release_date_us`: **the US theatrical date** (decision, Tanner 2026-11-24): The Numbers → RT theatrical → Box Office Mojo first day.
- `release_date`: US date where known, otherwise TMDB's (first release anywhere). `release_date_src` records which. `release_month` and `release_quarter` come from it.
- `release_date` is the **one** release date to use. Its two ingredients, `release_date_us` and `release_date_tmdb`, are kept in source detail for audits.
- `year` (TMDB) is **not changed**: the population and the train/test split stay as the contract defines them.

In [ ]:
combine("runtime_min", ["runtime", "rt_runtimeMinutes", "tn_Running Time (minutes)", "sl_runtime"])

dates = {c: pd.to_datetime(films[c], errors="coerce") for c in
         ["tn_Release Date", "rt_releaseDateTheaters", "bom_first_date", "release_date"]}
combine("release_date_us", ["tn_Release Date", "rt_releaseDateTheaters", "bom_first_date"], dates)
dates["release_date_us"] = films["release_date_us"]
films["release_date_tmdb"] = dates["release_date"]
combine("release_date", ["release_date_us", "release_date"], {"release_date_us": dates["release_date_us"], "release_date": dates["release_date"]})
films["release_date_src"] = films["release_date_src"].replace({"release_date_us": "us", "release_date": "tmdb_first_release"})
films["release_month"] = films["release_date"].dt.month
films["release_quarter"] = films["release_date"].dt.quarter
films = films.drop(columns="release_date_us_src")

## 5. Genre (C7)

TMDB genres (19 categories) first. RT and The Numbers genres fill only TMDB's blanks, translated to TMDB's names. Genres with no TMDB equivalent (*Biography*, *Sports*, *Holiday*, *LGBTQ+*, *Multiple Genres*) are left out of the fill.

In [ ]:
RT_GENRE = {"Action": "Action", "Adventure": "Adventure", "Animation": "Animation", "Anime": "Animation",
            "Comedy": "Comedy", "Stand-up": "Comedy", "Crime": "Crime", "Documentary": "Documentary", "Drama": "Drama",
            "Fantasy": "Fantasy", "History": "History", "Horror": "Horror", "Kids & family": "Family",
            "Music": "Music", "Musical": "Music", "Mystery & thriller": ["Mystery", "Thriller"], "Romance": "Romance",
            "Sci-fi": "Science Fiction", "War": "War", "Western": "Western"}
TN_GENRE = {"Action": "Action", "Adventure": "Adventure", "Black Comedy": "Comedy", "Comedy": "Comedy",
            "Concert/Performance": "Music", "Documentary": "Documentary", "Drama": "Drama", "Horror": "Horror",
            "Musical": "Music", "Reality": "Documentary", "Romantic Comedy": ["Romance", "Comedy"],
            "Thriller/Suspense": "Thriller", "Western": "Western"}


def translate(s, mapping):
    def one(lst):
        out = []
        for g in lst:
            m = mapping.get(g)
            out += m if isinstance(m, list) else ([m] if m else [])
        return out
    return join_list(split_list(s).map(one))


combine("genres_all", ["genres", "rt_genre", "tn_Genre"],
        {"genres": real("genres"), "rt_genre": translate(films["rt_genre"], RT_GENRE),
         "tn_Genre": translate(films["tn_Genre"].astype("string").str.strip(), TN_GENRE)})
TMDB_GENRES = set(g for l in split_list(films["genres"]) for g in l)
assert set(g for l in split_list(films["genres_all"]) for g in l) <= TMDB_GENRES

## 6. MPAA rating (C8)

The Numbers → RT → RT sl, standardized to **G / PG / PG-13 / R / NC-17 / NR** (sources agree on 99% of films). The Numbers' run-on text ("NC-17for explicit sexual content…") is cut to its leading rating; "Not Rated" and "NR" become NR; TV ratings and "Open" count as missing.

⚠️ RT never uses NR, so a blank RT rating can mean "never rated". `mpaa_rating` NR and missing are kept separate.

In [ ]:
def std_rating(s):
    s = s.astype("string").str.strip()
    m = s.str.extract(r"^(NC-?17|PG-13|PG|G|R|Not Rated|NR|Unrated)", expand=False)
    return m.replace({"NC17": "NC-17", "Not Rated": "NR", "Unrated": "NR"})


combine("mpaa_rating", ["tn_MPAA Rating", "rt_rating", "sl_content_rating"],
        {c: std_rating(films[c]) for c in ["tn_MPAA Rating", "rt_rating", "sl_content_rating"]})
films.loc[usable, "mpaa_rating"].value_counts(dropna=False)

## 7. People and companies (C9–C13)

Fill only: the second source fills the first source's blanks, with no name matching yet.
- `director`, `writer`: RT → RT sl. They're used as **track records from earlier films only**.
- `production_companies`: TMDB → The Numbers → RT sl.
- `distributor`: RT → Box Office Mojo.

**Cast (C11) is dropped as an input** (Tanner, 2026-11-24): only 5% coverage in 2020–23. `sl_actors` stays in source detail.

Company and distributor names differ by source ("Warner Bros." vs "Warner Bros. Pictures"). Standardizing them is a later cleaning step.

In [ ]:
combine("director", ["rt_director", "sl_directors"])
combine("writer", ["rt_writer", "sl_authors"])
combine("production_companies", ["production_companies", "tn_Production/Financing Companies", "sl_production_company"],
        {"production_companies": real("production_companies"),
         "tn_Production/Financing Companies": join_list(split_list(films["tn_Production/Financing Companies"])),
         "sl_production_company": real("sl_production_company")})
combine("distributor", ["rt_distributor", "bom_distributor"])

## 8. Country and language (C14–C15)

- `countries`: TMDB → The Numbers, with The Numbers' names translated to TMDB's ("United States" → "United States of America"). The Numbers writes Taiwan as "Taiwan, Province of China", which splits on the comma; the "Province of China" fragment is dropped.
- `original_language`: TMDB (ISO code) → RT (language name → code). `tn_Languages` lists every spoken language, not the original one, so it stays in source detail.

In [ ]:
TN_COUNTRY = {"United States": "United States of America", "Republic of Korea": "South Korea",
              "Russian Federation": "Russia", "Islamic Republic of Iran": "Iran",
              "Bolivarian Republic of Venezuela": "Venezuela", "Province of China": None}
tn_c = join_list(split_list(films["tn_Production Countries"]).map(
    lambda l: [TN_COUNTRY.get(c, c) for c in l if TN_COUNTRY.get(c, c)]))
combine("countries", ["production_countries", "tn_Production Countries"],
        {"production_countries": real("production_countries"), "tn_Production Countries": tn_c})

LANG = {"English": "en", "Hindi": "hi", "French": "fr", "Chinese": "zh", "Korean": "ko", "Spanish": "es",
        "Russian": "ru", "Japanese": "ja", "German": "de", "Italian": "it", "Arabic": "ar", "Norwegian": "no",
        "Tamil": "ta", "Danish": "da", "Telugu": "te", "Portuguese": "pt", "Turkish": "tr", "Polish": "pl",
        "Finnish": "fi", "Swedish": "sv", "Dutch": "nl", "Malayalam": "ml", "Hebrew": "he", "Romanian": "ro",
        "Czech": "cs", "Thai": "th", "Urdu": "ur", "Indonesian": "id", "Persian": "fa", "Malay": "ms",
        "Latin": "la", "Hungarian": "hu", "Afrikaans": "af", "Cantonese": "cn", "Greek": "el", "Icelandic": "is",
        "Vietnamese": "vi", "Bengali": "bn", "Kannada": "kn", "Marathi": "mr", "Punjabi": "pa", "Tagalog": "tl",
        "Ukrainian": "uk", "Serbian": "sr", "Croatian": "hr", "Estonian": "et", "Lithuanian": "lt", "Slovak": "sk"}
rt_lang = films["rt_originalLanguage"].astype("string").str.replace(r"\s*\(.*\)", "", regex=True).str.strip().map(LANG)
combine("original_language", ["original_language", "rt_originalLanguage"],
        {"original_language": real("original_language"), "rt_originalLanguage": rt_lang})

## 9. Review consensus and audience score (C16–C17): description only

Both are **post-release**, so they're never model inputs. They're kept to describe films and to show how reception relates to winners (Tanner, 2026-11-24).
- `critic_pct_fresh` (0–100): RT reviews % fresh → RT Tomatometer → RT sl Tomatometer.
- `audience_score` (0–100): TMDB average vote × 10 (only when the film has votes) → RT audience → RT sl audience.

In [ ]:
combine("critic_pct_fresh", ["rtrev_pct_fresh", "rt_tomatoMeter", "sl_tomatometer_rating"])
tmdb_aud = (films["vote_average"] * 10).where(films["vote_count"].gt(0))
combine("audience_score", ["vote_average", "rt_audienceScore", "sl_audience_rating"], {"vote_average": tmdb_aud})

## 10. Franchise, source material, creative type (C18)

These come from The Numbers only, so there's nothing to combine. Franchise names that were run together ("BatmanDark Knight Trilogy") are split at a lowercase→uppercase boundary into a `|`-separated list (U9). Names that are legitimately written that way (*MonsterVerse*, *xXx*) are protected.

In [ ]:
PROTECT = ["MonsterVerse", "xXx", "DreamWorks", "McG", "LEGO", "iCarly"]


def split_franchise(x):
    if pd.isna(x):
        return np.nan
    s = str(x)
    for i, p in enumerate(PROTECT):
        s = s.replace(p, f"\x00{i}\x00")
    s = __import__("re").sub(r"(?<=[a-z0-9’'\)])(?=[A-Z])", "|", s)
    for i, p in enumerate(PROTECT):
        s = s.replace(f"\x00{i}\x00", p)
    return s


films["franchise"] = films["tn_Franchise"].map(split_franchise)
films["source_material"] = real("tn_Source").astype("string").str.strip()
films["creative_type"] = real("tn_Creative Type").astype("string").str.strip()
changed = films.loc[films["franchise"].ne(films["tn_Franchise"]) & films["tn_Franchise"].notna(), ["tn_Franchise", "franchise"]]
print(f"Split {len(changed)} run-together franchise values (rows, all films)")
changed.drop_duplicates().head(10)

## 10b. Budget repair: when two budget sources disagree

`fin_budget` takes TMDB first. When TMDB's budget and The Numbers' (or Wikidata's) differ by more than 2×, TMDB is usually wrong (an extra zero: *Awake* $86M vs $8.6M), so The Numbers' value is used (Tanner, Edwards §8f).

In [ ]:
fin_budget_before = films["fin_budget"].copy()        # kept for the "revenue = budget" rule in §12
tmdb_b = real("budget")
other_b = real("tn_Production Budget (USD)").fillna(real("wd_wd_budget_usd"))
ratio = tmdb_b / other_b
fix = films["fin_budget_src"].eq("tmdb") & ((ratio > 2) | (ratio < 0.5))
films.loc[fix, "fin_budget"] = other_b[fix]
films.loc[fix, "fin_budget_src"] = "the_numbers_or_wikidata (TMDB differs > 2x)"
print(f"Budgets replaced: {fix.sum():,} films ({(fix & usable).sum():,} usable)")
films.loc[fix & usable, ["title", "year"]].assign(tmdb=tmdb_b[fix & usable], other=other_b[fix & usable]).sort_values("tmdb", ascending=False).head(8)

## 10c. Fill director, writer, runtime and source from Wikidata

Snapshot `data/raw/wikidata/wikidata_film_people_2026-11-24.csv` (`tools/get_wikidata_people.py`, matched by TMDB id). Tested on 200 films: director and writer agree with ours 96–97%, runtime within 5 min 96% (Edwards §9l). Distributor isn't used (Wikidata lists worldwide and streaming distributors).

Also: where our director (from Rotten Tomatoes) and Wikidata's share no name, our row picked up a different film with the same title (*A Christmas Carol* 2009 got a TV version's director). Wikidata's director replaces ours, and `director_mismatch` marks the film.

In [ ]:
import difflib, unicodedata

wd = pd.read_csv(CLEAN.parent / "raw" / "wikidata" / "wikidata_film_people_2026-11-24.csv").set_index("tmdb_id")
w = wd.reindex(films["id"].where(films["in_tmdb"]).astype("Int64")).set_index(films.index)


def names(s):
    return [t.strip() for t in str(s).replace("|", ",").split(",") if t.strip()] if pd.notna(s) else []


def norm(n):
    n = unicodedata.normalize("NFKD", n).encode("ascii", "ignore").decode()
    return "".join(ch for ch in n.lower() if ch.isalpha() or ch == " ").split()


def same_person(a, b):
    """Same name ignoring punctuation, accents and order; one name inside the other; same last name; or near-identical."""
    A, B = norm(a), norm(b)
    if not A or not B:
        return False
    if "".join(A) == "".join(B) or set(A) <= set(B) or set(B) <= set(A) or A[-1] == B[-1]:
        return True
    return difflib.SequenceMatcher(None, " ".join(sorted(A)), " ".join(sorted(B))).ratio() >= 0.8


both = films["director"].notna() & w["wd_director"].notna()
films["director_mismatch"] = both & pd.Series(
    [not any(same_person(x, y) for x in names(a) for y in names(b)) for a, b in zip(films["director"], w["wd_director"])],
    index=films.index)
films.loc[films["director_mismatch"], "director"] = w.loc[films["director_mismatch"], "wd_director"].str.replace("|", ", ")
films.loc[films["director_mismatch"], "director_src"] = "wikidata (replaced mismatch)"
print(f"Director replaced (mismatch with Wikidata): {(films['director_mismatch'] & usable).sum()} usable films")

for col, wcol in [("director", "wd_director"), ("writer", "wd_writer"), ("runtime_min", "wd_runtime_min")]:
    gap = films[col].isna() & w[wcol].notna()
    vals = w.loc[gap, wcol].str.replace("|", ", ") if w[wcol].dtype == object else w.loc[gap, wcol]
    films.loc[gap, col] = vals
    films.loc[gap, col + "_src"] = "wikidata"
    print(f"{col:<12} filled from Wikidata: {(gap & usable).sum():,} usable films")


def source_from_type(t):
    t = str(t).lower()
    for keys, val in [(("comic", "manga", "graphic novel"), "Based on Comic/Graphic Novel"),
                      (("video game",), "Based on Game"),
                      (("television series", "tv series", "television program"), "Based on TV"),
                      (("musical",), "Based on Musical or Opera"), (("play",), "Based on Play"),
                      (("fairy tale", "folk", "legend", "myth"), "Based on Folk Tale/Legend/Fairytale"),
                      (("film",), "Remake"),
                      (("novel", "short story", "novella", "book", "literary work", "written work"), "Based on Fiction Book/Short Story")]:
        if any(k in t for k in keys):
            return val
    return np.nan


films["source_material_src"] = pd.Series(np.where(films["source_material"].notna(), "the_numbers", pd.NA), index=films.index, dtype="string")
gap = films["source_material"].isna() & w["wd_based_on_type"].map(source_from_type).notna()
films.loc[gap, "source_material"] = w.loc[gap, "wd_based_on_type"].map(source_from_type)
films.loc[gap, "source_material_src"] = "wikidata"
print(f"source_material filled from Wikidata 'based on': {(gap & usable).sum():,} usable films")

## 10d. `source_group`: six source-material groups

Tanner's hypothesis: comic-book films vs original screenplays vs remakes matters. The Numbers' 23 categories are grouped into **Comic, Book, Real life, Remake, Other adaptation, Original**, filled from The Numbers → Wikidata (§10c) → TMDB keywords. Keywords aren't a model input (crowd-edited after release); here they only look up a fact known before production. Tested on 3,801 films: 74% agreement (Book 89%, Comic 72% and finds 93% of comic films, Original 72%, Remake 50%). `source_group_src` keeps the origin so the model can be rerun without the keyword values.

In [ ]:
def tn_group(s):
    if pd.isna(s):
        return np.nan
    if "Comic" in s: return "Comic"
    if s == "Remake": return "Remake"
    if "Real Life" in s or "Factual" in s: return "Real life"
    if "Fiction Book" in s: return "Book"
    if s == "Original Screenplay": return "Original"
    return "Other adaptation"   # TV, game, play, musical, folk tale, toy, spin-off...


def kw_group(s):
    if pd.isna(s) or not str(s).strip():
        return np.nan
    t = {x.strip() for x in str(s).lower().split(",")}
    has = lambda *ks: any(k in x for x in t for k in ks)
    if has("based on comic", "superhero", "based on graphic novel", "based on manga", "comic book", "based on webcomic", "based on cartoon"):
        return "Comic"
    if has("remake", "reboot", "based on movie", "based on short film"):
        return "Remake"
    if has("based on true story", "biography", "biographical", "based on memoir", "autobiograph"):
        return "Real life"
    if has("based on novel", "based on young adult novel", "based on children's book", "based on short story", "based on magazine", "literary adaptation"):
        return "Book"
    if has("based on video game", "based on tv series", "based on play", "based on toy", "based on fairy tale", "based on myths",
           "based on theme park", "based on song", "based on anime"):
        return "Other adaptation"
    return "Original"


sg = films["source_material"].map(tn_group)
kw = films["keywords"].map(kw_group)
films["source_group"] = sg.fillna(kw)
films["source_group_src"] = pd.Series(np.select([sg.notna(), kw.notna()], [films["source_material_src"].fillna(""), "tmdb_keywords"], ""),
                                      index=films.index).replace("", pd.NA).astype("string")
films.loc[usable, "source_group_src"].value_counts(dropna=False).rename("usable films").to_frame()

## 11. Missing flags (U6)

Every allowed or candidate input that isn't filled for all usable films gets `<column>_missing` (True = no source had a value). Models use the flag instead of a guessed value.

In [ ]:
INPUTS = ["runtime_min", "release_date", "genres_all", "mpaa_rating", "director", "writer", "production_companies",
          "distributor", "countries", "original_language", "franchise", "source_material", "creative_type", "source_group"]
flags = {}
for c in INPUTS:
    miss = films[c].isna()
    if miss[usable].any():
        films[c + "_missing"] = miss
        flags[c] = miss[usable].mean()
pd.Series(flags, name="share of usable films missing").map("{:.1%}".format).to_frame()

## 11b. Rule-based fills (after the missing flags)

Agreed by Tanner (Edwards §7i). The `_missing` flags above still record that the value was blank.
- **MPAA rating:** blank → `NR`. A blank often means the film wasn't rated in the US, which may itself be a signal.
- **Country:** blank → the main country of the original language (English → United States). Approximate; only a few films.
- **Franchise:** blank → `(none)`. Most films aren't in a franchise; The Numbers covers nearly all usable films.

In [ ]:
films["mpaa_rating"] = films["mpaa_rating"].fillna("NR")
LANG_COUNTRY = {"en": "United States of America", "fr": "France", "hi": "India", "ta": "India", "te": "India", "ml": "India",
                "kn": "India", "mr": "India", "bn": "India", "pa": "India", "es": "Spain", "ru": "Russia", "zh": "China",
                "cn": "Hong Kong", "ko": "South Korea", "ja": "Japan", "de": "Germany", "it": "Italy", "pt": "Brazil",
                "tr": "Turkey", "sv": "Sweden", "da": "Denmark", "no": "Norway", "fi": "Finland", "nl": "Netherlands",
                "pl": "Poland", "th": "Thailand", "fa": "Iran", "ar": "Egypt", "he": "Israel", "id": "Indonesia",
                "tl": "Philippines", "uk": "Ukraine", "cs": "Czech Republic", "ro": "Romania", "hu": "Hungary", "el": "Greece"}
gap = films["countries"].isna() & films["original_language"].map(LANG_COUNTRY).notna()
films.loc[gap, "countries"] = films.loc[gap, "original_language"].map(LANG_COUNTRY)
films.loc[gap, "countries_src"] = "from original_language"
films["franchise"] = films["franchise"].fillna("(none)")
print(f"Country filled from language: {(gap & usable).sum()} usable films")
films.loc[usable, ["mpaa_rating", "countries", "franchise"]].isna().sum().rename("still blank (usable)").to_frame()

## 12. Exclusion rules, population and label

Rules decided by Tanner in `Edwards-A10.ipynb` §8e and §8h (2026-11-24). A film gets the **first** reason that applies; nothing is deleted.

| Reason | Rule |
|---|---|
| `tmdb_only` | In none of The Numbers, Rotten Tomatoes (either dataset) or Box Office Mojo: budget and revenue can't be checked. Mostly junk entries. |
| `not_a_film` | *The Bear* (TV series, both entries), TMDB's *TV Movie* genre (no box office), runtime 400+ (TV series, video games) |
| `runtime_under_60` | Shorts, clips, episodes |
| `edition_or_cut` | Title says extended/special/ultimate edition, director's cut, version, making-of, stage play… (concert films stay: if people paid to see it in a theater, it's a movie) |
| `budget_error` | Over $5M per minute (real films top out near $2.5M; Edwards used $3M, raised after the budget repair pushed *The Force Awakens* to $3.9M), or under 70 min with a $20M+ budget (except *Winnie the Pooh* and *Pooh's Heffalump Movie*); 4 foreign budgets checked by hand (§8h) |
| `revenue_equals_budget` | Revenue exactly equals the budget: a copied value |
| `tiny_revenue_unrated` | Revenue under $10K and no MPAA rating (NR or blank) |

**Population (A7):** `usable` (2000–2023, budget ≥ $1M, revenue > 0), released in the US (has a US box office), and no `exclude_reason`.
**Label (A7, A9):** `winner` = US box office ≥ 1.25 × budget. `us_multiple` = US box office ÷ budget, for the 1× and 1.5× sensitivity checks. (`loser` = not winner is kept only so older exploration notebooks still run; don't use it.)

**Known gap:** a few real films are `tmdb_only` because the title matcher missed them in The Numbers (*F9* vs "F9: The Fast Saga", *The Butler* vs "Lee Daniels' The Butler"). They have no US box office in our data, so A7 excludes them either way; the fix belongs in the `00_build` matcher.

In [ ]:
n_sources = (films["tn_Movie URL"].notna().astype(int) + films["rt_id"].notna() + films["bom_title"].notna()
             + films["sl_rotten_tomatoes_link"].notna())
films["n_sources"] = n_sources
rt_ = films["runtime_min"]
genres = films["genres_all"].fillna("")
title = films["title"].fillna("")
rated = films["mpaa_rating_missing"].eq(False) & films["mpaa_rating"].isin(["G", "PG", "PG-13", "R", "NC-17"]) if "mpaa_rating_missing" in films \
        else films["mpaa_rating"].isin(["G", "PG", "PG-13", "R", "NC-17"])
edition = title.str.contains(r"\b(?:edition|extended|ultimate cut|director'?s cut|re-?release|original video|making of|"
                             r"behind the scenes|the play|broadway|season \d)\b", case=False, regex=True)
bpm = films["fin_budget"] / rt_
POOH = ["Winnie the Pooh", "Pooh's Heffalump Movie"]
WRONG_FOREIGN_BUDGET = [("Chasing the Dragon", 2017), ("A Aa", 2016), ("Pink", 2016), ("Burnout", 2014)]
wrong_foreign = pd.Series(False, index=films.index)
for t_, y_ in WRONG_FOREIGN_BUDGET:
    wrong_foreign |= title.eq(t_) & films["year"].eq(y_) & films["usable"]

rules = [
    ("tmdb_only", films["in_tmdb"] & n_sources.eq(0)),
    ("not_a_film", films["film_uid"].isin(["tmdb-1233925", "tmdb-1769662"]) | genres.str.contains("TV Movie") | rt_.ge(400)),
    ("runtime_under_60", rt_.lt(60)),
    ("edition_or_cut", edition),
    ("budget_error", (((bpm > 5e6) | (rt_.lt(70) & films["fin_budget"].ge(20e6))) & ~title.isin(POOH)) | wrong_foreign),
    ("revenue_equals_budget", films["fin_revenue"].eq(fin_budget_before)),
    ("tiny_revenue_unrated", films["fin_revenue"].lt(1e4) & ~rated),
]
reason = pd.Series(pd.NA, index=films.index, dtype="string")
for name, mask in reversed(rules):
    reason = reason.mask(mask.fillna(False), name)
films["exclude_reason"] = reason

films["in_population"] = films["usable"] & films["domestic_gross"].gt(0) & films["exclude_reason"].isna()
films["us_multiple"] = films["domestic_gross"] / films["fin_budget"]
films["winner"] = films["us_multiple"].ge(1.25).where(films["in_population"])   # THE label (A7 + A9)
films["loser"] = films["us_multiple"].lt(1.25).where(films["in_population"])    # old name, kept only so older exploration runs; don't use

print("Usable films by exclusion reason (first reason that applies):")
display(films.loc[usable, "exclude_reason"].fillna("(kept)").value_counts().rename("usable films").to_frame())
kept = usable & films["exclude_reason"].isna()
print(f"Usable {usable.sum():,} -> after rules {kept.sum():,} -> released in the US (population) {films['in_population'].sum():,}")
pop_ = films[films["in_population"]]
print(f"Winners (US box office >= 1.25x budget): {pop_['winner'].mean():.1%}; "
      f"at 1x: {pop_['us_multiple'].ge(1).mean():.1%}; at 1.5x: {pop_['us_multiple'].ge(1.5).mean():.1%}")

## 12b. Engineered features (winners from `Edwards-A10.ipynb` §10–16, 2026-11-26)

Each feature uses only information known **before the film's release** (A6). Track records use only **earlier** films' results.

| Feature | What | Evidence (training years) |
|---|---|---|
| `log_budget` | log10 of budget | budget's effect is curved; the model handles the bend |
| `primary_genre` | first genre listed | +0.077 AUC over budget |
| `origin_group` | US only / US co-production / non-US | 1.02× vs 0.70× vs 0.11× median return |
| `sequel` | an earlier franchise film was already released, **or** the title carries an installment number (§13 rule v2) | 3.25× the odds of winning after period and budget |
| `distributor_group` | Major / Mini-major / Specialty / Other / Blank, after merging name variants | +0.169 (re-test without Blank) |
| `major_producer` | a major studio among the production companies | +0.040 |
| `release_weekday` | Wednesday / Friday / other | +0.020 |
| `director_prior_films`, `director_prior_hit_rate` | director's earlier films, and the share that were winners (blank = no scored history) | 0.469 → 0.586 AUC with budget |
| `writer_…`, `company_…` | same for writers and production companies | Blumhouse 7.2× (§16b) |
| `rating_overlap_competition` | films in the same or a neighboring rating (G↔PG↔PG-13↔R) opening within ±6 weeks, ÷ what's typical for the rating | +0.012, lower return in every rating |

In [ ]:
import re

feat = films  # add columns in place
feat["log_budget"] = np.log10(films["fin_budget"].where(films["fin_budget"] > 0))
feat["primary_genre"] = films["genres_all"].fillna("").str.split(",").str[0].str.strip().replace("", np.nan)
ctry = films["countries"].fillna("")
feat["origin_group"] = np.select([ctry.eq("United States of America"), ctry.str.contains("United States")],
                                 ["US only", "US co-production"], "non-US")
feat.loc[ctry.eq(""), "origin_group"] = np.nan
rd = pd.to_datetime(films["release_date"], format="mixed", errors="coerce")
feat["release_weekday"] = rd.dt.day_name().where(lambda s: s.isin(["Wednesday", "Friday"]), "Other day").where(rd.notna())

# Distributor: merge name variants of the big studios, then group
STUDIO = {"Warner Bros.": "Warner Bros.", "Warner Bros. Pictures": "Warner Bros.", "20th Century Fox": "20th Century Fox",
          "Fox": "20th Century Fox", "Twentieth Century Fox": "20th Century Fox", "20th Century Studios": "20th Century Fox",
          "Universal Pictures": "Universal", "Universal Studios": "Universal", "Universal": "Universal",
          "Walt Disney": "Disney", "Walt Disney Pictures": "Disney", "Buena Vista Pictures": "Disney", "Buena Vista": "Disney",
          "Touchstone Pictures": "Disney", "Walt Disney Studios Motion Pictures": "Disney",
          "Sony Pictures Entertainment": "Sony", "Columbia Pictures": "Sony", "Sony Pictures": "Sony", "Sony Pictures Releasing": "Sony",
          "Paramount Pictures": "Paramount", "Paramount": "Paramount", "Lionsgate Films": "Lionsgate", "Lionsgate": "Lionsgate",
          "Lions Gate Films": "Lionsgate"}
MAJORS = {"Disney", "Warner Bros.", "Universal", "Sony", "Paramount", "20th Century Fox"}
MINI = {"Lionsgate", "Weinstein Co.", "Miramax Films", "New Line Cinema", "DreamWorks SKG", "Summit Entertainment", "Relativity Media",
        "Metro-Goldwyn-Mayer", "STX Entertainment", "Open Road", "FilmDistrict", "CBS Films", "Screen Gems", "Dimension Films"}
SPECIALTY = {"Sony Pictures Classics", "Fox Searchlight", "Focus Features", "IFC Films", "Magnolia Pictures", "Roadside Attractions", "A24",
             "Samuel Goldwyn Company", "ThinkFilm", "Warner Independent", "Paramount Vantage", "First Look", "Anchor Bay",
             "Freestyle Releasing", "Overture Films", "Artisan Entertainment", "Lionsgate Premiere", "Neon"}
d1 = films["distributor"].str.split(",").str[0].str.strip()
feat["distributor_studio"] = d1.map(STUDIO).fillna(d1)
feat["distributor_group"] = np.select([feat["distributor_studio"].isin(MAJORS), feat["distributor_studio"].isin(MINI),
                                       feat["distributor_studio"].isin(SPECIALTY), feat["distributor_studio"].isna()],
                                      ["Major", "Mini-major", "Specialty", "Blank"], "Other")
feat["major_producer"] = films["production_companies"].fillna("").str.lower().str.contains(
    r"warner|universal|walt disney|disney|pixar|columbia|sony pictures|paramount|20th century|twentieth century|fox").astype(int)
print(films.loc[films["in_population"], ["primary_genre", "origin_group", "release_weekday", "distributor_group"]].nunique().to_dict())
films.loc[films["in_population"], "distributor_group"].value_counts()

In [ ]:
# Sequel: earlier franchise film already released (in our data), or an installment number in the title (rule v2)
real_films = films["usable"] & films["exclude_reason"].isna()
fr = films.loc[real_films & films["franchise"].ne("(none)"), ["franchise"]].assign(rd=rd[real_films])
first_release = fr.assign(f=fr["franchise"].str.split("|")).explode("f").groupby("f")["rd"].min()
prior_in_data = pd.Series([fs != "(none)" and any(first_release.get(f, pd.NaT) < d for f in str(fs).split("|"))
                           for fs, d in zip(films["franchise"], rd)], index=films.index)

ROMAN = {"II": 2, "III": 3, "IV": 4, "V": 5, "VI": 6, "VII": 7, "VIII": 8, "IX": 9, "X": 10}
WORDS = {"two": 2, "three": 3, "four": 4, "five": 5, "second": 2, "third": 3, "fourth": 4}
PATS = [r"\b(?:part|chapter|episode|vol\.?|volume)\s+(\d+|[ivx]+|two|three|four|five)\b",
        r"(?<!^)\b(\d)\b(?!\d)(?=\s*(?::|-|–|$|\())",
        r"\b(II|III|IV|VI|VII|VIII|IX)\b(?=\s*(?::|-|–|$|\())"]


def installment(title):
    for pat in PATS:
        m = re.search(pat, str(title).strip(), flags=re.IGNORECASE if "part" in pat else 0)
        if m:
            v = m.group(1)
            n = int(v) if v.isdigit() else ROMAN.get(v.upper(), WORDS.get(v.lower()))
            if n and 2 <= n <= 12:
                return n
    return np.nan


def base_title(t):
    t = re.sub(r"[:\-–(].*$", "", str(t))
    t = re.sub(r"\b(part|chapter|episode|vol\.?|volume)\b.*$", "", t, flags=re.I)
    return re.sub(r"\s+(\d+|II|III|IV|VI|VII|VIII|IX)\s*$", "", t.strip()).strip().lower()


tm = pd.read_csv(CLEAN.parent / "raw" / "TMDB_movie_dataset_v11.csv", usecols=["title", "release_date", "vote_count"], low_memory=False)
tm = tm[tm["vote_count"].fillna(0) >= 20]
earliest = tm.assign(b=tm["title"].str.lower().str.strip(), d=pd.to_datetime(tm["release_date"], errors="coerce")).groupby("b")["d"].min()
num = films["title"].map(installment)
explicit = films["title"].fillna("").str.contains(r"\b(?:part|chapter|episode|vol\.?|volume)\s+(?:\d+|[ivx]+|two|three|four|five)\b", case=False, regex=True) \
           | films["title"].fillna("").str.contains(r"\b(?:II|III|IV|VI|VII|VIII|IX)\b", regex=True)
earlier_base = pd.Series([pd.notna(earliest.get(b, pd.NaT)) and earliest.get(b) < d for b, d in zip(films["title"].map(base_title), rd)], index=films.index)
title_sequel = num.notna() & (explicit | films["franchise"].ne("(none)") | earlier_base)
feat["sequel"] = (prior_in_data | title_sequel).astype(int)
feat["sequel_src"] = np.select([prior_in_data, title_sequel], ["franchise_in_data", "title"], "none")
print(f"Sequels in the population: {feat.loc[films['in_population'], 'sequel'].sum():,} "
      f"({feat.loc[films['in_population'], 'sequel_src'].value_counts().to_dict()})")

In [ ]:
# Track records: earlier films only. "Hit" = US box office >= 1.25x budget (known only for population films).
order = films.loc[real_films, ["film_uid"]].assign(rd=rd[real_films]).dropna().sort_values("rd")
hit = films["us_multiple"].ge(1.25).astype(float).where(films["in_population"])


def people(s):
    return [t.strip().lower() for t in str(s).split(",") if t.strip()] if pd.notna(s) else []


def track_record(col):
    hist, n_prev, rate = {}, {}, {}
    for uid in order["film_uid"]:
        i = uid_index[uid]
        names = people(films.at[i, col])
        prev = [x for nm in names for x in hist.get(nm, [])]
        scored = [x for x in prev if not np.isnan(x)]
        n_prev[uid], rate[uid] = len(prev), (np.mean(scored) if scored else np.nan)
        for nm in names:
            hist.setdefault(nm, []).append(hit[i])
    return films["film_uid"].map(n_prev), films["film_uid"].map(rate)


uid_index = pd.Series(films.index, index=films["film_uid"])
for col, name in [("director", "director"), ("writer", "writer"), ("production_companies", "company")]:
    feat[f"{name}_prior_films"], feat[f"{name}_prior_hit_rate"] = track_record(col)
pop_ = films["in_population"]
films.loc[pop_, ["director_prior_films", "director_prior_hit_rate", "writer_prior_hit_rate", "company_prior_hit_rate"]].describe().round(2)

In [ ]:
# Competition from the same or neighboring ratings within +-6 weeks, relative to what's typical for the rating
ADJ = {"G": {"G", "PG"}, "PG": {"G", "PG", "PG-13"}, "PG-13": {"PG", "PG-13", "R"}, "R": {"PG-13", "R", "NC-17"},
       "NC-17": {"R", "NC-17"}, "NR": {"NR"}}
pool = films.loc[real_films & films["release_date_src"].eq("us"), ["film_uid", "mpaa_rating"]].assign(rd=rd).dropna(subset=["rd"]).sort_values("rd")
pd_days = pool["rd"].values.astype("datetime64[D]").astype(np.int64)
p_uid, p_rat = pool["film_uid"].values, pool["mpaa_rating"].fillna("NR").values
counts = pd.Series(np.nan, index=films.index)
for i in films.index[films["usable"] & rd.notna()]:
    d = np.datetime64(rd[i], "D").astype(np.int64)
    lo, hi = np.searchsorted(pd_days, d - 42), np.searchsorted(pd_days, d + 42, side="right")
    r_ = films.at[i, "mpaa_rating"] if pd.notna(films.at[i, "mpaa_rating"]) else "NR"
    counts[i] = sum((u != films.at[i, "film_uid"]) and (rr in ADJ.get(r_, {r_})) for u, rr in zip(p_uid[lo:hi], p_rat[lo:hi]))
feat["rating_overlap_count"] = counts
typical = counts[films["in_population"] & films["year"].le(2015)].groupby(films["mpaa_rating"]).median()   # training years only
feat["rating_overlap_competition"] = counts / films["mpaa_rating"].map(typical).replace(0, 1)
films.loc[pop_].groupby("mpaa_rating")["rating_overlap_count"].median()

## 13. Column groups and save

Columns are reordered into groups, and every column's group and role is written to `films_clean_columns.csv`:

| Group | What | Model input? |
|---|---|---|
| `key` | `film_uid`, ids, titles | no |
| `population` | `year`, `usable`, `in_tmdb`, `merged_ids`, `n_sources`, `exclude_reason`, `director_mismatch`, **`in_population`** | no (defines rows and splits) |
| `money` | `fin_budget` (input); **label (A7, A9):** `domestic_gross` (US box office), `us_multiple`, **`winner`** (`loser` = old name, don't use); `fin_revenue` (worldwide, checks only) | budget only |
| `input` | the combined columns above, `source_group`, plus `_missing` flags | yes (A4/A6: anything controlled before release) |
| `provenance` | `_src` columns | no |
| `description` | `critic_pct_fresh`, `audience_score`, review counts, critic consensus text | no (post-release) |
| `source detail` | every original source column | no |

In [ ]:
KEY = ["film_uid", "id", "imdb_id", "title", "original_title"]
POPULATION = ["year", "usable", "in_tmdb", "merged_ids", "n_sources", "exclude_reason", "director_mismatch", "in_population"]
MONEY = ["fin_budget", "fin_budget_src", "domestic_gross", "us_multiple", "winner", "loser", "fin_revenue", "fin_revenue_src"]
INPUT = ["runtime_min", "release_date", "release_month", "release_quarter",
         "genres_all", "mpaa_rating", "director", "writer", "production_companies", "distributor", "countries",
         "original_language", "franchise", "source_material", "source_group", "creative_type",
         # engineered (§12b)
         "log_budget", "primary_genre", "origin_group", "sequel", "distributor_studio", "distributor_group", "major_producer",
         "release_weekday", "director_prior_films", "director_prior_hit_rate", "writer_prior_films", "writer_prior_hit_rate",
         "company_prior_films", "company_prior_hit_rate", "rating_overlap_count", "rating_overlap_competition"]
INPUT += [c + "_missing" for c in INPUTS if c + "_missing" in films]
PROVENANCE = [c for c in films.columns if c.endswith("_src") and c not in MONEY]
DESCRIPTION = ["critic_pct_fresh", "audience_score", "rtrev_n_reviews", "rtrev_n_top_critic", "rtrev_n_before_release",
               "slrev_n_reviews", "slrev_n_before_release", "sl_critics_consensus"]
groups = {"key": KEY, "population": POPULATION, "money": MONEY, "input": INPUT,
          "provenance": PROVENANCE, "description": DESCRIPTION}
placed = [c for cols in groups.values() for c in cols]
groups["source detail"] = [c for c in films.columns if c not in placed]
assert len(set(placed)) == len(placed) and all(c in films for c in placed)

def role(c, g):
    if g == "input":
        return "allowed input"
    if c == "fin_budget":
        return "allowed input"
    if c in ("domestic_gross", "us_multiple", "winner"):
        return "label (outcome, A7/A9)"
    if c == "loser":
        return "old label name (= not winner); don't use"
    if c.endswith("_src"):
        return "build metadata"
    return {"key": "key", "population": "defines rows / splits", "money": "post-release: checks only",
            "provenance": "build metadata", "description": "post-release: description only",
            "source detail": "not used (audit only)"}[g]

dictionary = pd.DataFrame([{"column": c, "group": g, "role": role(c, g),
                            "% filled (usable films)": round(100 * films.loc[usable, c].notna().mean(), 1),
                            "% filled (population)": round(100 * films.loc[films["in_population"], c].notna().mean(), 1)}
                           for g, cols in groups.items() for c in cols])
clean = films[dictionary["column"].tolist()]
assert len(clean) == n_rows and clean["film_uid"].is_unique
clean.to_csv(CLEAN / "films_clean.csv", index=False)
dictionary.to_csv(CLEAN / "films_clean_columns.csv", index=False)
print(f"films_clean.csv: {len(clean):,} films x {clean.shape[1]} columns ({clean['usable'].sum():,} usable, {clean['in_population'].sum():,} in population)")
dictionary.groupby("group", sort=False).size().rename("columns").to_frame()

In [ ]:
with pd.option_context("display.max_rows", 80):
    display(dictionary[dictionary["group"].isin(["money", "input", "description"])])